In [2]:
!pip -q install geopandas folium branca rapidfuzz openpyxl requests

In [3]:
import io
import re
import json
import requests
import pandas as pd
import geopandas as gpd

from rapidfuzz import process, fuzz

import folium
from folium import GeoJson
from folium.features import GeoJsonTooltip
from branca.colormap import LinearColormap

In [ ]:
GEOJSON_PATH = "INDIAN_SUB_DISTRICTS.geojson"
india_gdf = gpd.read_file(GEOJSON_PATH)

print(india_gdf.head())

   OBJECTID stcode11 dtcode11 sdtcode11  Shape_Length    Shape_Area  \
0      4808       35      638     05916  5.206834e+04  1.295483e+08   
1      4809       35      638     05917  4.928417e+05  4.692465e+08   
2      4810       35      638     05918  3.536977e+05  1.120545e+09   
3      4811       35      639     05919  1.127189e+06  1.428804e+09   
4      4812       35      639     05920  5.380326e+05  8.030010e+08   

              stname                   dtname        sdtname  Subdt_LGD  \
0  ANDAMAN & NICOBAR                 Nicobars    Car Nicobar     5916.0   
1  ANDAMAN & NICOBAR                 Nicobars       Nancowry     5917.0   
2  ANDAMAN & NICOBAR                 Nicobars  Great Nicobar     5918.0   
3  ANDAMAN & NICOBAR  North  & Middle Andaman       Diglipur     5919.0   
4  ANDAMAN & NICOBAR  North  & Middle Andaman     Mayabunder     5920.0   

   Dist_LGD  State_LGD                                           geometry  
0     603.0         35  POLYGON ((92.77359 9.2

In [ ]:
state_name_col = "stname"

maharashtra_gdf = india_gdf[
    india_gdf[state_name_col].str.lower() == "maharashtra"
]

print(maharashtra_gdf.head())
print(f"Number of sub-districts in Maharashtra: {len(maharashtra_gdf)}")

      OBJECTID stcode11 dtcode11 sdtcode11   Shape_Length    Shape_Area  \
3123      3969       27      497     03950  229977.366751  1.084474e+09   
3124      3970       27      497     03951  206186.733326  1.498276e+09   
3125      3971       27      497     03952  162472.061389  5.141909e+08   
3126      3972       27      497     03953  195674.992974  1.405330e+09   
3127      3973       27      497     03954  212746.249040  1.216660e+09   

           stname     dtname    sdtname  Subdt_LGD  Dist_LGD  State_LGD  \
3123  MAHARASHTRA  Nandurbar  Akkalkuwa     3950.0     486.0         27   
3124  MAHARASHTRA  Nandurbar     Akrani     3951.0     486.0         27   
3125  MAHARASHTRA  Nandurbar     Talode     3952.0     486.0         27   
3126  MAHARASHTRA  Nandurbar    Shahade     3953.0     486.0         27   
3127  MAHARASHTRA  Nandurbar  Nandurbar     3954.0     486.0         27   

                                               geometry  
3123  POLYGON ((74.12203 21.68172, 74.11

In [29]:
sub_district_name_col = "sdtname"
census_data_path = "POP_AREA.xlsx"

census_df = pd.read_excel(
    census_data_path,
    sheet_name="Sheet1",
    skiprows=1
    )

census_df.drop(census_df.index[:2], inplace=True)

census_df.rename(columns={
    'Total/\nRural/\nUrban': "Total/Rural/Urban",
    'India/ State/ Union Territory/ District/ Sub-district': "Administrative Unit",
    "Number of villages": "Number of Inhabited villages",
    "Unnamed: 7": "Number of Uninhabited villages",
    "Population": "Total Population",
    "Unnamed: 11": "Male Population",
    "Unnamed: 12": "Female Population",
    "State  Code": "State Code",
}, inplace=True)

print(census_df.columns.tolist())
print(census_df.head())
print(f"Number of rows in census data: {len(census_df)}")

['State Code', 'District Code', 'Sub District Code', 'Administrative Unit', 'Name', 'Total/Rural/Urban', 'Number of Inhabited villages', 'Number of Uninhabited villages', 'Number of towns', 'Number of households', 'Total Population', 'Male Population', 'Female Population', 'Area\n (In sq. km)', 'Population per sq. km.']
   State Code District Code Sub District Code Administrative Unit  \
2         0.0           000             00000               INDIA   
3         0.0           000             00000               INDIA   
4         0.0           000             00000               INDIA   
5         1.0           000             00000               STATE   
6         1.0           000             00000               STATE   

                 Name Total/Rural/Urban Number of Inhabited villages  \
2            INDIA @&             Total                       597608   
3             INDIA $             Rural                       597608   
4             INDIA $             Urban        

In [ ]:
census_df = census_df[census_df["Total/Rural/Urban"].str.lower() == "total"]
census_df = census_df[census_df["Administrative Unit"].str.lower() == "sub-district"]
maharashtra_census_df = census_df[census_df["State Code"] == 27]
print(maharashtra_census_df.head())

print(f"Number of rows in maharashtra census data: {len(maharashtra_census_df)}")


    State Code District Code Sub District Code Administrative Unit      Name  \
11         1.0           001             00001        SUB-DISTRICT   Kupwara   
14         1.0           001             00002        SUB-DISTRICT  Handwara   
17         1.0           001             00003        SUB-DISTRICT    Karnah   
23         1.0           002             00004        SUB-DISTRICT      Khag   
26         1.0           002             00005        SUB-DISTRICT   Beerwah   

   Total/Rural/Urban Number of Inhabited villages  \
11             Total                          118   
14             Total                          196   
17             Total                           39   
23             Total                           49   
26             Total                          102   

   Number of Uninhabited villages  Number of towns  Number of households  \
11                              4              7.0               63022.0   
14                              3              1